# NeuralGeoQA — Stage 3: Evaluation Pipeline

Loads the trained **GeoQABERT** model and runs the full 4-stage pipeline:
1. BERT inference (span + relation + qtype)
2. Wikidata entity linking
3. SPARQL query construction
4. Execution + answer formatting

Requires `qtype_model_v2/` in Google Drive (trained in the training notebook or separately).

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers torch fuzzywuzzy python-Levenshtein requests pandas tqdm scikit-learn

In [ ]:
# Cell 2 — Mount Drive + paths
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR   = '/content/drive/MyDrive/triple_head/'
MODEL_DIR  = os.path.join(BASE_DIR, 'GEO/qtype_model_v2')
TEST_FILE  = os.path.join(BASE_DIR, 'GEO/geo_test_201.tsv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'GEO/evaluation')
CACHE_DIR  = os.path.join(BASE_DIR, 'GEO/cache')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,  exist_ok=True)

assert os.path.exists(MODEL_DIR), f'Model not found: {MODEL_DIR}'
assert os.path.exists(TEST_FILE), f'Test file not found: {TEST_FILE}'
print('Paths OK.')

In [ ]:
# Cell 3 — Model definition + loading

import json, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoConfig

class GeoQABERT(nn.Module):
    def __init__(self, model_name, num_relations, num_qtypes, dropout=0.1):
        super().__init__()
        self.config           = AutoConfig.from_pretrained(model_name)
        self.encoder          = AutoModel.from_pretrained(model_name)
        hidden                = self.config.hidden_size
        self.span_start       = nn.Linear(hidden, 1)
        self.span_end         = nn.Linear(hidden, 1)
        self.rel_dropout      = nn.Dropout(dropout)
        self.rel_classifier   = nn.Linear(hidden, num_relations)
        self.qtype_dropout    = nn.Dropout(dropout)
        self.qtype_classifier = nn.Linear(hidden, num_qtypes)
        self.loss_fn          = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, qtype_labels=None):
        out     = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq_out = out.last_hidden_state
        cls_out = seq_out[:, 0, :]
        return {
            'loss':          self.loss_fn(self.qtype_classifier(self.qtype_dropout(cls_out)), qtype_labels)
                             if qtype_labels is not None else None,
            'start_logits':  self.span_start(seq_out).squeeze(-1),
            'end_logits':    self.span_end(seq_out).squeeze(-1),
            'rel_logits':    self.rel_classifier(self.rel_dropout(cls_out)),
            'qtype_logits':  self.qtype_classifier(self.qtype_dropout(cls_out)),
        }

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(os.path.join(MODEL_DIR, 'config.json')) as f:
    model_cfg = json.load(f)

model = GeoQABERT(
    model_name=model_cfg['model_name'],
    num_relations=model_cfg['num_relations'],
    num_qtypes=model_cfg['num_qtypes'],
)
state = torch.load(os.path.join(MODEL_DIR, 'best_model.pt'), map_location=DEVICE, weights_only=True)
model.load_state_dict(state, strict=False)
model.to(DEVICE).eval()

tokenizer     = AutoTokenizer.from_pretrained(os.path.join(MODEL_DIR, 'tokenizer'), use_fast=True)
id_to_qtype   = {int(k): v for k, v in model_cfg['id_to_qtype'].items()}
id_to_relation = {int(k): v for k, v in model_cfg.get('id_to_relation', {}).items()}
MAX_LEN       = model_cfg.get('max_length', 96)

print(f'Model on {DEVICE}  qtypes={len(id_to_qtype)}  relations={len(id_to_relation)}')

In [ ]:
# Cell 4 — BERT inference + QType override
import re

def refine_qtype(merged, question):
    ql = question.lower()
    if merged == 'B_spatial' and re.search(r'\b(north|south|east|west|n[eo]|s[eo])\s+of\b', ql):
        return 'B_directional'
    if merged == 'C_class':
        return 'C_class_spatial_rel' if any(k in ql for k in ['border','cross','flow','discharge','run through']) else 'C_class_in'
    if merged == 'E_class_near' and (re.search(r'\d+\s*(km|kilometer|meter|m\b|mile)', ql) or any(k in ql for k in ['at most','within','range of','radius'])):
        return 'E_class_distance'
    if merged == 'G_superlative' and any(k in ql for k in ['bigger than','larger than','longer than','taller than']):
        return 'G_comparative'
    return merged

def override_qtype(bert_qtype, question):
    ql = question.lower()
    if bert_qtype == 'B_spatial':
        if re.search(r'\b(north|south|east|west|ne|nw|se|sw)\s+of\b', ql): return 'B_directional'
        if not re.search(r'\b(near|km|close|within|range|away|radius|there|any)', ql): return 'B_boolean'
    if bert_qtype == 'E_class_near' and not re.search(r'\b(near|close|km|within|away|radius|at most|nearest|closest)', ql):
        return 'C_class_spatial_rel' if re.search(r'\b(border|cross|flow|discharge|run through)', ql) else 'C_class_in'
    return refine_qtype(bert_qtype, question)

@torch.no_grad()
def bert_predict(question):
    enc  = tokenizer(str(question), add_special_tokens=True, max_length=MAX_LEN,
                     padding='max_length', truncation=True, return_tensors='pt')
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)
    out  = model(input_ids=ids, attention_mask=mask)

    si = int(torch.argmax(out['start_logits'], dim=1).item())
    ei = int(torch.argmax(out['end_logits'],   dim=1).item())
    if ei < si: ei = si
    span_text = re.sub(r'[?.,!;:]+$', '', tokenizer.decode(ids[0,si:ei+1], skip_special_tokens=True)).strip()

    rel_probs = torch.softmax(out['rel_logits'], dim=1)[0]
    topk      = torch.topk(rel_probs, min(3, rel_probs.size(0)))
    relations = []
    for i, p in zip(topk.indices, topk.values):
        rid = id_to_relation.get(int(i), str(int(i)))
        m   = re.match(r'^R(\d+)$', str(rid))
        if m: rid = f'P{m.group(1)}'
        relations.append({'id': rid, 'prob': float(p)})

    qtype_probs = torch.softmax(out['qtype_logits'], dim=1)[0]
    qi          = int(torch.argmax(qtype_probs).item())
    bert_qtype  = id_to_qtype[qi]

    return {
        'span_text':        span_text,
        'relations':        relations,
        'qtype':            override_qtype(bert_qtype, question),
        'qtype_bert_raw':   bert_qtype,
        'qtype_confidence': float(qtype_probs[qi]),
    }

# Quick diagnostic
DIAG = [
    ('Which rivers are in Wales?',           'C_class_in'),
    ('How many counties does England have?', 'G_count'),
    ('Is Liverpool part of Scotland?',       'B_boolean'),
    ('Is Hampshire north of Berkshire?',     'B_directional'),
    ('Which mountains in Scotland have height more than 1000 meters?', 'F_thematic_spatial'),
]
correct = 0
for q, expected in DIAG:
    r = bert_predict(q)
    ok = r['qtype'] == expected
    if ok: correct += 1
    print(f'{'✓' if ok else '✗'} {q[:55]:<57} → {r["qtype"]}')
print(f'Diagnostic: {correct}/{len(DIAG)}')

In [ ]:
# Cell 5 — Wikidata cache + entity linking

import os, json, time, hashlib, re, requests
from fuzzywuzzy import fuzz

WIKIDATA_API    = 'https://www.wikidata.org/w/api.php'
WIKIDATA_SPARQL = 'https://query.wikidata.org/sparql'
WD_HEADERS      = {'User-Agent': 'NeuralGeoQA/1.0'}

_memory_cache = {}
_cache_file   = os.path.join(CACHE_DIR, 'wikidata_cache.json')
_last_api     = _last_sparql = 0.0

if os.path.exists(_cache_file):
    try:
        with open(_cache_file) as f: _memory_cache = json.load(f)
        print(f'Cache: {len(_memory_cache)} entries')
    except: pass

def _ckey(p,v): return f'{p}:{hashlib.md5(v.encode()).hexdigest()[:12]}'
def _cget(p,v): return _memory_cache.get(_ckey(p,v))
def _cset(p,v,r): _memory_cache[_ckey(p,v)] = r

def _rl(sparql=False):
    global _last_api, _last_sparql
    limit = 1.5 if sparql else 0.8
    ref   = _last_sparql if sparql else _last_api
    wait  = limit - (time.time() - ref)
    if wait > 0: time.sleep(wait)
    if sparql: _last_sparql = time.time()
    else: _last_api = time.time()

def _get(url, params, sparql=False, retries=3, timeout=20):
    for attempt in range(retries):
        _rl(sparql)
        try:
            r = requests.get(url, params=params, headers=WD_HEADERS, timeout=timeout)
            if r.status_code == 429: time.sleep(5*(attempt+1)); continue
            r.raise_for_status(); return r
        except requests.exceptions.Timeout:
            if attempt < retries-1: time.sleep(3)
    return None

def wd_search(query, limit=10):
    c = _cget('search', query)
    if c is not None: return c
    r = _get(WIKIDATA_API, {'action':'wbsearchentities','search':query,'language':'en','limit':limit,'format':'json'})
    if r is None: return wd_search_sparql(query, limit)
    try:
        res = r.json().get('search',[])
        _cset('search', query, res); return res
    except: return wd_search_sparql(query, limit)

def wd_search_sparql(query, limit=10):
    c = _cget('sparql_search', query)
    if c is not None: return c
    sparql = f'''SELECT ?item ?itemLabel ?itemDescription WHERE {{
      SERVICE wikibase:mwapi {{
        bd:serviceParam wikibase:api "EntitySearch" .
        bd:serviceParam wikibase:endpoint "www.wikidata.org" .
        bd:serviceParam mwapi:search "{query}" .
        bd:serviceParam mwapi:language "en" .
        ?item wikibase:apiOutputItem mwapi:item .
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }} LIMIT {limit}'''
    r = _get(WIKIDATA_SPARQL, {'query':sparql,'format':'json'}, sparql=True)
    if r is None: return []
    try:
        res = [{'id':b['item']['value'].split('/')[-1],
                'label':b.get('itemLabel',{}).get('value',query),
                'description':b.get('itemDescription',{}).get('value','')}
               for b in r.json().get('results',{}).get('bindings',[])]
        _cset('sparql_search', query, res); return res
    except: return []

GEO_KW = {'city','town','village','county','river','lake','mountain','castle','bridge',
           'park','church','museum','station','airport','district','region','country',
           'island','forest','england','scotland','wales','ireland','london','greece'}
STOP   = {'which','what','where','how','is','does','are','do','the','a','an','there',
           'that','through','in','many','has','have','near','from','more','less','than',
           'most','any','can','at','and','not','no','with','on','to','by','for','or'}

def clean_span(s):
    s = re.sub(r'[?.,!;:]+$','',s.strip()).strip()
    return s.title() if s and s[0].islower() else s

def make_variants(span):
    variants = [span]
    for pat in [r'^(?:county|county of|the county of)\s+', r'^(?:river|the river)\s+',
                r'^(?:the|a|an)\s+', r'^(?:mount|lake|loch)\s+']:
        s = re.sub(pat,'',span,flags=re.I).strip()
        if s and s != span: variants.append(s)
    if not span.lower().startswith('river') and len(span.split())==1: variants.append(f'River {span}')
    seen=set(); return [v for v in variants if v.lower() not in seen and not seen.add(v.lower())]

def wd_details_batch(qids):
    results,uncached = {},{}
    for qid in qids:
        c = _cget('details',qid)
        if c: results[qid]=c
        else: uncached[qid]=True
    if not uncached: return results
    vals   = ' '.join(f'wd:{q}' for q in list(uncached)[:5])
    sparql = f'''SELECT ?entity ?typeLabel ?coord ?adminLabel ?countryLabel WHERE {{
      VALUES ?entity {{ {vals} }}
      OPTIONAL {{ ?entity wdt:P31 ?type }}
      OPTIONAL {{ ?entity wdt:P625 ?coord }}
      OPTIONAL {{ ?entity wdt:P131 ?admin }}
      OPTIONAL {{ ?entity wdt:P17 ?country }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" }}
    }}'''
    for qid in uncached:
        results.setdefault(qid,{'types':[],'coordinates':None,'admin':None,'country':None})
    r = _get(WIKIDATA_SPARQL,{'query':sparql,'format':'json'},sparql=True,timeout=25)
    if r:
        try:
            for b in r.json().get('results',{}).get('bindings',[]):
                qid = b['entity']['value'].split('/')[-1]
                d   = results.setdefault(qid,{'types':[],'coordinates':None,'admin':None,'country':None})
                if 'typeLabel' in b and b['typeLabel']['value'] not in d['types']: d['types'].append(b['typeLabel']['value'])
                if 'coord' in b and not d['coordinates']: d['coordinates']=b['coord']['value']
                if 'adminLabel' in b and not d['admin']: d['admin']=b['adminLabel']['value']
                if 'countryLabel' in b and not d['country']: d['country']=b['countryLabel']['value']
        except: pass
    for qid,d in results.items(): _cset('details',qid,d)
    return results

def score_candidates(candidates, span, question=''):
    sl = span.lower()
    for c in candidates:
        label = c.get('label','')
        desc  = c.get('description','').lower()
        sim   = max(fuzz.ratio(sl,label.lower()),fuzz.partial_ratio(sl,label.lower()),
                    fuzz.token_sort_ratio(sl,label.lower()))/100.0
        c['score'] = round(sim + (0.1 if any(k in desc for k in GEO_KW) else 0)
                           + (0.15 if sl==label.lower() else 0), 4)
    candidates.sort(key=lambda c:c['score'],reverse=True)
    return candidates

def enrich(candidates, top_k=5):
    top     = candidates[:top_k]
    details = wd_details_batch([c['id'] for c in top])
    for c in top:
        d = details.get(c['id'],{'types':[],'coordinates':None,'admin':None,'country':None})
        c['details'] = d; c['has_coords']=bool(d['coordinates'])
        c['types']=d['types']; c['country']=d['country']
        if d['coordinates']: c['score']+=0.15
    candidates.sort(key=lambda c:c['score'],reverse=True)
    return candidates

def link_entity(span, question='', top_k=3):
    if not span or not span.strip(): return []
    cs = clean_span(span)
    all_cands = {}
    for v in make_variants(cs):
        for c in wd_search(v):
            all_cands.setdefault(c['id'],c)
    if not all_cands: return []
    cands = score_candidates(list(all_cands.values()), cs, question)
    cands = enrich(cands, top_k=min(top_k+2,5))
    return cands[:top_k]

def extract_secondary(question, primary_span):
    pl,words,entities,i = (primary_span or '').lower(),question.split(),[],1
    while i < len(words):
        w = re.sub(r'[?.,!;:]+$','',words[i])
        if w and w[0].isupper() and w.lower() not in STOP:
            ew,j = [w],i+1
            while j<len(words):
                nw=re.sub(r'[?.,!;:]+$','',words[j])
                if nw and nw[0].isupper() and nw.lower() not in STOP: ew.append(nw);j+=1
                elif nw.lower() in ('of','the','and','in') and j+1<len(words) and words[j+1][0].isupper(): ew.append(nw);j+=1
                else: break
            name=' '.join(ew)
            if name.lower()!=pl and pl not in name.lower() and name.lower() not in pl and len(name)>1:
                entities.append(name)
            i=j
        else: i+=1
    return entities

def link_all_entities(question, span, qtype=''):
    result={'primary':None,'secondary':[],'distance_km':None,'numeric_constraint':None}
    if span and span.strip():
        cands=link_entity(span,question,top_k=3)
        if cands:
            b=cands[0]
            result['primary']={'qid':b['id'],'label':b['label'],'score':b['score'],
                               'coordinates':b.get('details',{}).get('coordinates'),
                               'country':b.get('country'),'types':b.get('types',[]),
                               'all_candidates':cands}
    for sp in extract_secondary(question,span)[:2]:
        c=link_entity(sp,question,top_k=1)
        if c: result['secondary'].append({'qid':c[0]['id'],'label':c[0]['label'],'span':sp,'score':c[0]['score']})
    dm=re.search(r'(\d+(?:\.\d+)?)\s*(km|kilometer|meter|m\b|mile)',question,re.I)
    if dm:
        val,unit=float(dm.group(1)),dm.group(2).lower()
        if unit in ('m','meter'): val/=1000
        elif unit=='mile': val*=1.609
        result['distance_km']=val
    nm=re.search(r'(more than|over|less than|at least|at most|taller than|bigger than)\s+(\d+)',question,re.I)
    if nm: result['numeric_constraint']=f'{nm.group(1)} {nm.group(2)}'
    return result

print('Entity linking ready.')

In [ ]:
# Cell 6 — SPARQL execution, query builder, answer formatter, full pipeline
# (Import from evaluate.py or run inline)

# Paste the build_query, execute_sparql, format_answer, detect_answer_type,
# and answer_question functions from evaluate.py here, OR:

import sys
sys.path.insert(0, '/content/drive/MyDrive/triple_head/')  # if evaluate.py is on Drive
# from evaluate import build_query, execute_sparql, format_answer, detect_answer_type, answer_question

# --- If running inline, paste the functions below ---
# (build_query, execute_sparql, format_answer, detect_answer_type, answer_question)
# All are in evaluate.py in the repo.

print('Pipeline functions loaded. Run Cell 7 to start evaluation.')

In [ ]:
# Cell 7 — Quick pipeline demo (5 questions)
DEMO_QUESTIONS = [
    'Which rivers are in Wales?',
    'Is Liverpool part of Scotland?',
    'How many counties does England have?',
    'Which counties border county Lincolnshire?',
    'Is Hampshire north of Berkshire?',
]

for q in DEMO_QUESTIONS:
    print(f'\n{"-"*60}')
    r = answer_question(q, verbose=True)
    print(f'  ANSWER: {r["answer"]}')

In [ ]:
# Cell 8 — Full evaluation on 201 test questions

import pandas as pd, time, json
from tqdm import tqdm

df = pd.read_csv(TEST_FILE, sep='\t')
print(f'Loaded {len(df)} test questions')

results = []
for i, row in tqdm(df.iterrows(), total=len(df), desc='Evaluating'):
    q       = row['Question']
    gold_qt = row.get('QType', '')
    r       = answer_question(q, verbose=False)
    results.append({
        'TestID':       row.get('TestID', i),
        'Question':     q,
        'Gold_QType':   gold_qt,
        'Pred_QType':   r['bert']['qtype'],
        'QType_Match':  r['bert']['qtype'] == gold_qt if gold_qt else None,
        'Pred_Span':    r['bert']['span_text'],
        'Entity_QID':   r['linking']['primary']['qid'] if r['linking']['primary'] else '',
        'Entity_Label': r['linking']['primary']['label'] if r['linking']['primary'] else '',
        'Template':     r.get('template',''),
        'Success':      r.get('success', False),
        'N_Results':    r.get('n_results', 0),
        'Pred_Answer':  str(r.get('answer','')),
        'Gold_Answer':  str(row.get('Gold_Answer','')),
    })
    time.sleep(0.3)

n          = len(results)
qt_matches = sum(bool(r['QType_Match']) for r in results if r['QType_Match'] is not None)
qt_total   = sum(1 for r in results if r['QType_Match'] is not None)
n_success  = sum(r['Success'] for r in results)

print(f'\n{"="*60}')
print(f'  QType accuracy:   {qt_matches}/{qt_total} ({100*qt_matches/max(1,qt_total):.1f}%)')
print(f'  Pipeline success: {n_success}/{n} ({100*n_success/n:.1f}%)')

results_df = pd.DataFrame(results)
out_path   = os.path.join(OUTPUT_DIR, 'answer_evaluation.tsv')
results_df.to_csv(out_path, sep='\t', index=False)

# Save cache
with open(_cache_file,'w') as f: json.dump(_memory_cache,f)
print(f'Results saved: {out_path}')
print(f'Cache saved: {len(_memory_cache)} entries')